In [ ]:
import pandas as pd
import re
import csv
import re
import sys
from collections import Counter, defaultdict

df_extract = pd.read_csv(
    "../data/interim/1_2_extract_15_16_concat.csv",
    low_memory=False,
)

df_ND1516 = pd.read_csv(
    "../data/raw/test_regards_citoyens/ND15+16_interventions_hemicycle_rich.tsv",
    sep="\t",
    engine="python",
    on_bad_lines="warn",
)

(1127829, 37)

## Comparaison pnum vs id_syceron

In [83]:
# === Comparaison simple par id_syceron ===

# TODO pourrait changer le nom puisque pas forcéement p
# extraite equivalent id_syceron vs url Pnum (pas toujours le P)
P_NUMBER_RE = re.compile(r"#P?(\d+)", re.I)


def extract_pnum(url):
    if not url or pd.isna(url):
        return None
    url = str(url).strip()
    m = P_NUMBER_RE.search(url)
    return m.group(1) if m else None


# Colonnes normalisées
df_extract["id_syceron"] = pd.to_numeric(
    df_extract["id_syceron"], errors="raise"
).astype("Int64")
df_ND1516["pnum"] = pd.to_numeric(
    df_ND1516["source"].apply(extract_pnum), errors="raise"
).astype("Int64")


# Comparaison sur les colonnes normalisées
ids_extract = set(df_extract["id_syceron"].dropna().astype(int))
ids_ND = set(df_ND1516["pnum"].dropna().astype(int))

common = ids_extract & ids_ND
only_extract = ids_extract - ids_ND
only_ND = ids_ND - ids_extract


print("=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===")
print(f"IDs communs              : {len(common):>10,}")
print(f"Uniquement dans extract  : {len(only_extract):>10,}")
print(f"Uniquement dans ND15-16  : {len(only_ND):>10,}")

print(f"\nTotal IDs extract : {len(ids_extract):>10,}")
print(f"Total IDs ND      : {len(ids_ND):>10,}")

=== RÉSUMÉ GLOBAL (par id_syceron uniquement) ===
IDs communs              :  1,065,985
Uniquement dans extract  :     61,477
Uniquement dans ND15-16  :     43,516

Total IDs extract :  1,127,462
Total IDs ND      :  1,109,501


In [67]:
# Vérification sur un échantillon aléatoire des IDs communs
sample_common = pd.Series(list(common)).sample(10)

df_sample_extract = df_extract[df_extract["id_syceron"].isin(sample_common)][
    ["id_syceron", "texte"]
].rename(columns={"id_syceron": "id"})  # ← adapter si la colonne texte a un autre nom

df_sample_ND = df_ND1516[df_ND1516["pnum"].isin(sample_common)][
    ["pnum", "intervention"]
].rename(columns={"pnum": "id"})  # ← adapter si la colonne intervention a un autre nom

# Fusion sur l'id commun
df_check = df_sample_extract.merge(df_sample_ND, on="id")

display(df_check)

print(
    "LÉO, NORMAL QUE TU AIES DES 'DOUBLONS'",
    "\nTON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)",
)

,id,texte,intervention
0,1027602,Députés d’une majorité territoriale autonomist...,<p>Députés d'une majorité territoriale autonom...
1,1337542,Je voudrais rectifier quelques éléments. Tout ...,<p>Je voudrais rectifier quelques éléments. To...
2,1337542,Je voudrais rectifier quelques éléments. Tout ...,<p>Applaudissements sur les bancs du groupe La...
3,1337542,Je voudrais rectifier quelques éléments. Tout ...,<p>Je peux concevoir que l'on dise…</p>
4,1337542,Je voudrais rectifier quelques éléments. Tout ...,<p>Exclamations sur les bancs du groupe FI. </p>
5,1485731,"Monsieur le président, monsieur le ministre, m...","<p>Monsieur le président, monsieur le ministre..."
6,1485731,"Monsieur le président, monsieur le ministre, m...",<p>Applaudissements sur plusieurs bancs des gr...
7,1656407,Il en est de même de l’amendement no 637.,<p>Il en est de même de l'amendement no 637.</p>
8,1656407,Il en est de même de l’amendement no 637.,<p>L'amendement no 637 est retiré. </p>
9,1934340,« Démembrement » serait le mot juste !,<p>« Démembrement » serait le mot juste !</p>


LÉO, NORMAL QUE TU AIES DES 'DOUBLONS' 
TON FICHIER MARCHE C'EST À CAUSE  DU CHANGEMENT TEXTE DANS ND)


In [ ]:
# repérage et exports
df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)]
df_only_ND.to_csv("only_ND.csv", index=False)

df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)]
df_only_extract.to_csv("only_extract.csv", index=False)

# filtrer pour cas pas intervenants

# cas ND
# Vérifie les cas où ni "parlementaire" ni "personnalite" ne sont renseignés
mask_no_speaker = df_only_ND["parlementaire"].fillna("").astype(str).str.strip().eq(
    ""
) & df_only_ND["personnalite"].fillna("").astype(str).str.strip().eq("")

df_only_ND_no_speaker = df_only_ND[mask_no_speaker]
df_only_ND_with_speaker = df_only_ND[~mask_no_speaker]


print(
    f"df_only_ND : Lignes sans parlementaire ET sans personnalite : {len(df_only_ND_no_speaker):,} / {len(df_only_ND):,}"
)

df_only_ND_with_speaker.to_csv("only_ND_with_speaker.csv", index=False)

# cas extract
# Vérifie les cas sans infos orateur dans extract
mask_no_speaker_extract = (
    df_only_extract["id_acteur"].isna()
    & df_only_extract["nom_orateur"].isna()
    & df_only_extract["id_orateur"].isna()
)

df_only_extract_no_speaker = df_only_extract[mask_no_speaker_extract]
df_only_extract_with_speaker = df_only_extract[~mask_no_speaker_extract]

print(
    f"df_only_extract_no_speaker : Lignes sans id_acteur ET sans nom_orateur ET sans id_orateur: {len(df_only_extract_no_speaker):,} / {len(df_only_extract):,}"
)

df_only_extract_with_speaker.to_csv("only_extract_with_speaker.csv", index=False)

df_only_ND : Lignes sans parlementaire ET sans personnalite : 44,570 / 45,408
df_only_extract_no_speaker : Lignes sans id_acteur ET sans nom_orateur ET sans id_orateur: 60,274 / 61,843


## Recherche texte pnum/id "absents" vs fichier opposé

In [ ]:
import unicodedata
import re
import html


def normalize_text(s):
    if pd.isna(s):
        return ""


def normalize_text(texte):
    if not isinstance(texte, str):
        return ""
    # Normaliser les caractères Unicode
    texte = unicodedata.normalize("NFC", texte)
    # Décoder les entités HTML
    texte = html.unescape(texte)
    # Supprimer les balises HTML/XML > espace (éviter collage de mots)
    texte = re.sub(r"<[^>]+>", " ", texte)
    # Supprimer contenu entre parenthèses
    # NOTE : CHOIX FORT SELON CE QUI VEUT ÊTRE ÉTUDIÉ
    # Supprime des didascalies ("Applaudissements", etc.)
    # mais aussi tout autre contenu entre parenthèses
    # ne gère pas les parenthèses imbriquées mais sont extrêmement rares (parfois sur (e))
    # texte = re.sub(r"\([^()]*\)", "", texte)
    # Uniformiser apostrophes (utile pour regex)
    texte = texte.replace("’", "'").replace("\u02bc", "'")
    # Normaliser les espaces (après unescape(), couvre \xa0, \t, \n)
    # et supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()
    # BONUS
    texte = str(texte).lower()

    return texte


# colonnes normalisées
df_ND1516["intervention_norm"] = (
    df_ND1516.get("intervention", "").fillna("").apply(normalize_text)
)
df_extract["texte_norm"] = df_extract.get("texte", "").fillna("").apply(normalize_text)

# recréer les sous-ensembles et limiter aux cas avec orateur
df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)].copy()
df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)].copy()

mask_ND_speaker = df_only_ND["parlementaire"].fillna("").astype(str).str.strip().ne(
    ""
) | df_only_ND["personnalite"].fillna("").astype(str).str.strip().ne("")
df_only_ND_with_speaker = df_only_ND[mask_ND_speaker].copy()

mask_extract_speaker = (
    df_only_extract["id_acteur"].notna()
    | df_only_extract["id_orateur"].notna()
    | df_only_extract["nom_orateur"].fillna("").astype(str).str.strip().ne("")
)
df_only_extract_with_speaker = df_only_extract[mask_extract_speaker].copy()


def search_snippets_fast(
    source_df, source_text_col, big_target, snippet_len=150, limit_rows=None
):
    rows = source_df if limit_rows is None else source_df.head(limit_rows)
    results = []
    for idx, row in rows.iterrows():
        snippet = row.get(source_text_col, "")[:snippet_len]
        found = bool(snippet) and (snippet in big_target)
        results.append({"index": idx, "snippet": snippet, "found": found})
    return pd.DataFrame(results)


# recherche rapide (snippet in big_text)
big_ND = " ".join(df_ND1516["intervention_norm"].dropna().tolist())
big_extract = " ".join(df_extract["texte_norm"].dropna().tolist())

fast_res_extract_in_ND = search_snippets_fast(
    df_only_extract_with_speaker, "texte_norm", big_ND, snippet_len=150, limit_rows=None
)
fast_res_ND_in_extract = search_snippets_fast(
    df_only_ND_with_speaker,
    "intervention_norm",
    big_extract,
    snippet_len=150,
    limit_rows=None,
)

fast_res_extract_in_ND.to_csv(
    "fast_only_extract_with_speaker_search_in_ND.csv", index=False
)
fast_res_ND_in_extract.to_csv(
    "fast_only_ND_with_speaker_search_in_extract.csv", index=False
)

print(
    "fast_only_extract_with_speaker -> found:",
    fast_res_extract_in_ND["found"].sum(),
    "/",
    len(fast_res_extract_in_ND),
)
print(
    "fast_only_ND_with_speaker     -> found:",
    fast_res_ND_in_extract["found"].sum(),
    "/",
    len(fast_res_ND_in_extract),
)


fast_only_extract_with_speaker -> found: 899 / 1569
fast_only_ND_with_speaker     -> found: 382 / 838


In [ ]:
# TODO : aller creuser les cas pas trouver pour voir

## EXPLORATION

In [100]:
df_only_ND_with_speaker["source"].value_counts()

source
http://www.assemblee-nationale.fr/15/cri/congres/20184001.asp#P1360145            92
http://www.assemblee-nationale.fr/15/cri/congres/20184001.asp#P1360418            19
http://www.assemblee-nationale.fr/15/cri/congres/20184001.asp#P1360406            11
http://www.assemblee-nationale.fr/15/cri/congres/20184001.asp#P1360290            10
http://www.assemblee-nationale.fr/15/cri/congres/20184001.asp#P1360520             7
                                                                                  ..
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181023.asp#P1380534     1
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181022.asp#P1380344     1
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181008.asp#P1362579     1
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181007.asp#P1361447     1
https://www.assemblee-nationale.fr/16/cri/2023-2024/20240235.asp#3507303           1
Name: count, Length: 604, dtype: int64

In [114]:
df_only_ND_with_speaker["source_short"] = (
    df_only_ND_with_speaker["source"].str.split("#").str[0]
)
df_only_ND_with_speaker["source_short"].value_counts().to_csv(
    "only_ND_with_speaker_source_short_counts.csv", index=True
)

/var/folders/rq/xsj46x_s2rg87wdksm1_jl3c0000gn/T/ipykernel_71239/825431286.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_only_ND_with_speaker["source_short"] = (


In [ ]:
df_only_ND_with_speaker[
    df_only_ND_with_speaker["source_short"].str.contains("congres", case=False)
]["source_short"].value_counts()

In [ ]:
df_only_ND_with_speaker[
    df_only_ND_with_speaker["source_short"].str.contains("extra") == 1
]["source_short"].value_counts()

source_short
http://www.assemblee-nationale.fr/15/cri/2016-2017-extra2/20172001.asp     10
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181040.asp       8
https://www.assemblee-nationale.fr/16/cri/2021-2022-extra/20221020.asp      6
https://www.assemblee-nationale.fr/16/cri/2021-2022-extra/20221019.asp      5
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181024.asp       5
http://www.assemblee-nationale.fr/15/cri/2016-2017-extra/20171010.asp       4
http://www.assemblee-nationale.fr/15/cri/2018-2019-extra2/20192012.asp      4
https://www.assemblee-nationale.fr/16/cri/2022-2023-extra2/20232012.asp     3
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181023.asp       3
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181041.asp       3
http://www.assemblee-nationale.fr/15/cri/2017-2018-extra/20181027.asp       3
https://www.assemblee-nationale.fr/16/cri/2021-2022-extra/20221009.asp      3
https://www.assemblee-nationale.fr/16/cri/2022-2023

In [104]:
df_only_ND_with_speaker["source_short"] = df_only_ND_with_speaker["source"].str[:-8]
df_only_ND_with_speaker["source_short"].value_counts().to_csv(
    "only_ND_with_speaker_source_short_counts.csv", index=True
)

In [162]:
fast_res_extract_in_ND

,index,snippet,found
0,903,je mets aux voix l'ensemble du projet de loi.,True
1,1669,"mes chers collègues, je précise que nous avons...",False
2,2935,"(la séance, suspendue à seize heures dix, est ...",False
3,2945,"la parole est à m. patrick hetzel, pour souten...",True
4,3918,cet amendement vise à substituer à un décret e...,True
...,...,...,...
1564,1126740,"je suis saisie de trois amendements, nos 3194,...",False
1565,1127123,nous en venons à une nouvelle série d'amendeme...,False
1566,1127400,cependant il ne s'agit pas seulement de repére...,False
1567,1127446,nous souhaitons que la décision d'accorder ou ...,False


In [161]:
df_only_extract_with_speaker[["id_acteur", "id_orateur", "nom_orateur", "texte"]]

,id_acteur,id_orateur,nom_orateur,texte
903,PA332747,332747.0,M. le président,Je mets aux voix l’ensemble du projet de loi.
1669,PA332747,332747.0,M. le président,"Mes chers collègues, je précise que nous avons..."
2935,PA720746,NaN,NaN,"(La séance, suspendue à seize heures dix, est ..."
2945,PA720746,720746.0,M. le président,"La parole est à M. Patrick Hetzel, pour souten..."
3918,PA720512,720512.0,M. Laurent Pietraszewski,Cet amendement vise à substituer à un décret e...
...,...,...,...,...
1126740,PA721908,721908.0,Mme la présidente,"Je suis saisie de trois amendements, nos 3194,..."
1127123,PA721908,721908.0,Mme la présidente,Nous en venons à une nouvelle série d’amendeme...
1127400,PA793876,793876.0,Mme Laurence Cristol,Cependant il ne s’agit pas seulement de repére...
1127446,PA642847,642847.0,M. Thibault Bazin,Nous souhaitons que la décision d’accorder ou ...


In [ ]:
(
    df_only_extract_with_speaker["id_acteur"].isna().sum(),
    df_only_extract_with_speaker["id_orateur"].isna().sum(),
    df_only_extract_with_speaker["nom_orateur"].isna().sum(),
)

(2, 531, 375)

In [ ]:
test = df_only_extract_with_speaker[
    df_only_extract_with_speaker["id_orateur"].isna() == 1
]

In [152]:
test[["id_acteur", "id_orateur", "nom_orateur", "texte"]]

,id_acteur,id_orateur,nom_orateur,texte
2935,PA720746,NaN,NaN,"(La séance, suspendue à seize heures dix, est ..."
4381,PA722150,NaN,NaN,(M. Sacha Houlié remplace M. François de Rugy ...
6322,PA722150,NaN,NaN,"(La séance, suspendue à seize heures dix, est ..."
7242,PA720622,NaN,NaN,(Mme Carole Bureau-Bonnard remplace M. Françoi...
8458,PA332747,NaN,NaN,"(La séance, suspendue à seize heures trente-ci..."
...,...,...,...,...
1088442,PA794354,NaN,NaN,Ce débat a été demandé par le groupe La France...
1091366,PA609590,NaN,NaN,Je vais donner la parole à chacun de nos invit...
1096555,PA794354,NaN,NaN,"Ce débat, organisé à la demande du groupe Soci..."
1096610,PA794354,NaN,NaN,La parole est à Mme la secrétaire d’État charg...


In [ ]:
Madame la ministre, je profite de votre prése

In [ ]:
# TODO cibler et comparer uniquement pour ceux avec orateurs vs le rest

In [ ]:
# # =========================
# # recréer les sous-ensembles pour qu'ils contiennent les colonnes normalisées
# df_only_ND = df_ND1516[df_ND1516["pnum"].isin(only_ND)].copy()
# df_only_extract = df_extract[df_extract["id_syceron"].isin(only_extract)].copy()

# def search_snippets(source_df, source_text_col, target_series, snippet_len=150, limit_rows=None):
#     rows = source_df if limit_rows is None else source_df.head(limit_rows)
#     results = []
#     for idx, row in rows.iterrows():
#         snippet = row.get(source_text_col, "")
#         if not snippet:
#             results.append({"index": idx, "snippet": "", "found": False, "count": 0, "matches_idx": ""})
#             continue
#         snippet = snippet[:snippet_len]
#         pattern = re.escape(snippet)
#         matches = target_series.str.contains(pattern, regex=True, na=False)
#         match_idx = matches[matches].index.tolist()
#         results.append(
#             {
#                 "index": idx,
#                 "snippet": snippet,
#                 "found": len(match_idx) > 0,
#                 "count": len(match_idx),
#                 "matches_idx": ",".join(map(str, match_idx[:10])),
#             }
#         )
#     return pd.DataFrame(results)

# # concaténation rapide des textes cibles
# big_ND = " ".join(df_ND1516["intervention_norm"].dropna().tolist())
# big_extract = " ".join(df_extract["texte_norm"].dropna().tolist())

# def search_snippets_fast(source_df, source_text_col, big_target, snippet_len=150, limit_rows=None):
#     rows = source_df if limit_rows is None else source_df.head(limit_rows)
#     results = []
#     for idx, row in rows.iterrows():
#         snippet = row.get(source_text_col, "")[:snippet_len]
#         found = bool(snippet) and (snippet in big_target)
#         results.append({"index": idx, "snippet": snippet, "found": found})
#     return pd.DataFrame(results)

# # usage rapide
# fast_res_extract_in_ND = search_snippets_fast(df_only_extract, "texte_norm", big_ND, snippet_len=150, limit_rows=500)
# fast_res_ND_in_extract = search_snippets_fast(df_only_ND, "intervention_norm", big_extract, snippet_len=150, limit_rows=500)
# fast_res_extract_in_ND.to_csv("fast_only_extract_search_in_ND.csv", index=False)
# fast_res_ND_in_extract.to_csv("fast_only_ND_search_in_extract.csv", index=False)

# # affichage rapide des cas trouvés / non trouvés
# print("fast_only_extract -> found:", fast_res_extract_in_ND["found"].sum(), " / ", len(fast_res_extract_in_ND))
# print("fast_only_ND     -> found:", fast_res_ND_in_extract["found"].sum(), " / ", len(fast_res_ND_in_extract))


# # Pour tout, mais long
# # # Cherche extraits de df_only_extract dans df_ND1516.intervention
# # res_extract_in_ND = search_snippets(df_only_extract, "texte_norm", df_ND1516["intervention_norm"], snippet_len=150, limit_rows=100)
# # res_extract_in_ND.to_csv("only_extract_search_in_ND.csv", index=False)

# # # Cherche extraits de df_only_ND dans df_extract.texte
# # res_ND_in_extract = search_snippets(df_only_ND, "intervention_norm", df_extract["texte_norm"], snippet_len=150, limit_rows=100)
# # res_ND_in_extract.to_csv("only_ND_search_in_extract.csv", index=False)

# # # affichage rapide des cas trouvés / non trouvés
# # print("only_extract -> found:", res_extract_in_ND["found"].sum(), " / ", len(res_extract_in_ND))
# # print("only_ND     -> found:", res_ND_in_extract["found"].sum(), " / ", len(res_ND_in_extract))


In [ ]:
def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    # Normaliser les caractères Unicode
    texte = unicodedata.normalize("NFC", texte)
    # Décoder les entités HTML
    texte = html.unescape(texte)
    # Supprimer les balises HTML/XML > espace (éviter collage de mots)
    texte = re.sub(r"<[^>]+>", " ", texte)
    # Supprimer contenu entre parenthèses
    # NOTE : CHOIX FORT SELON CE QUI VEUT ÊTRE ÉTUDIÉ
    # Supprime des didascalies ("Applaudissements", etc.)
    # mais aussi tout autre contenu entre parenthèses
    # ne gère pas les parenthèses imbriquées mais sont extrêmement rares (parfois sur (e))
    texte = re.sub(r"\([^()]*\)", "", texte)
    # Uniformiser apostrophes (utile pour regex)
    texte = texte.replace("’", "'").replace("\u02bc", "'")
    # Normaliser les espaces (après unescape(), couvre \xa0, \t, \n)
    # et supprimer les espaces multiples
    texte = re.sub(r"\s+", " ", texte).strip()

    return texte